## TechMind — Mejora del Modelo: Más Ejemplos en Categorías con F1 Bajo

#### Equipo tejONEs

#### 08_mejora_modelo.ipynb

- El proceso general que sigue este notebook es:
    - [x] Ítem 1: sumar más ejemplos en las categorías con F1 bajo (Backend, Cloud, Data Science, Frontend)
    - [x] Ítem 2: reentrenar con pipeline idéntico al baseline y comparar métricas
    - [x] Sanity check: reproducir ≈0.71 sobre el unificado oficial (valida el arnés)
    - [x] Exportación de artefactos con nombres nuevos (sin pisar el baseline)

### Diseño del experimento (para revisión del equipo)
1. El dataset ampliado **no agrega filas al unificado oficial**: se reconstruye desde
   las cuatro fuentes finales corregidas (Coursera v2, OpenAlex v2, MSLearn y
   StackExchange corregido), elevando el tope de 200 a 250 solo en las categorías
   con F1 bajo. Representa el dataset que el equipo adoptará como oficial tras la
   promoción de los v2, más ejemplos extra en las categorías débiles.
2. El pipeline es idéntico al baseline (limpieza canónica del PR #5, mismo TF-IDF,
   misma semilla 42, misma Regresión Logística): cualquier diferencia en métricas
   se debe al dato, no al método.
3. Limitación conocida: Frontend no mejora (0.72 → 0.72); su confusión viene de
   React/React Native (Mobile) y temas web compartidos, no de falta de ejemplos.

##Importaciones y limpieza

In [ ]:
import re, unicodedata, joblib
import pandas as pd
from pathlib import Path
from html import unescape
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, f1_score

try:
    from shared.limpieza_texto import limpiar_texto   # para correr desde el repo
    print("✅ Usando shared.limpieza_texto (canónica)")
except ImportError:
    import nltk; nltk.download('stopwords', quiet=True)
    STOP = set(stopwords.words('spanish'))
    def limpiar_texto(texto):                        # réplica idéntica para Colab
        texto = re.sub(r'<[^>]+>', ' ', unescape(str(texto)))
        sin_p = ''.join(' ' if unicodedata.category(c).startswith('P') else c
                        for c in texto.lower())
        return ' '.join(p for p in sin_p.split() if p not in STOP)
    print("✅ Usando réplica inline de la limpieza canónica")

✅ Usando réplica inline de la limpieza canónica


##Cargar datos (unificado oficial + pool de fuentes):

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE = Path('/content/drive/MyDrive/Datasets_TechMind')
EQ = DRIVE / 'Semana 1 - Dataset final - Repo GitHub/procesados'
ENTREGA = DRIVE / '1 Semana - con el Dataset final con todo incluido - Repo GitHub/procesados'

df_actual = pd.read_csv(ENTREGA / 'dataset_FINAL_UNIFICADO_techmind.csv')  # con el que se entrenó el baseline

df_pool = pd.concat([
    pd.read_csv(DRIVE / 'procesados/dataset_FINAL_coursera.csv'),
    pd.read_csv(EQ / 'dataset_FINAL_mslearn.csv'),
    pd.read_csv(EQ / 'dataset_FINAL_openalex.csv'),
    pd.read_csv(DRIVE / 'Code_Laura/Datos_procesados/dataset_FINAL_stackexchange_corregido.csv'),
], ignore_index=True)

print("Unificado oficial:", len(df_actual)); print(df_actual['categoria'].value_counts())
print("\nPool:", len(df_pool)); print(df_pool['categoria'].value_counts())



from google.colab import drive
drive.mount('/content/drive')

DRIVE = Path('/content/drive/MyDrive/Datasets_TechMind')

# Carpeta oficial final
F_FINAL = DRIVE / '1 Semana - con el Dataset final con todo incluido - Repo GitHub/procesados'

# 1. Unificado oficial para el Sanity Check
df_actual = pd.read_csv(F_FINAL / 'dataset_FINAL_UNIFICADO_techmind.csv')

# 2. Pool de fuentes para el modelo mejorado
# (Ahora TODAS leen de la carpeta oficial F_FINAL, excepto StackExchange que está en la carpeta de Laura)
df_pool = pd.concat([
    pd.read_csv(F_FINAL / 'dataset_FINAL_coursera.csv'),    # Tu Coursera v2 limpio
    pd.read_csv(F_FINAL / 'dataset_FINAL_mslearn.csv'),     # MSLearn oficial
    pd.read_csv(F_FINAL / 'dataset_FINAL_openalex.csv'),    # Tu OpenAlex v2 limpio
    pd.read_csv(DRIVE / 'Code_Laura/Datos_procesados/dataset_FINAL_stackexchange_corregido.csv'), # SE corregido
], ignore_index=True)

print("Unificado oficial:", len(df_actual))
print(df_actual['categoria'].value_counts())

print("\nPool de fuentes:", len(df_pool))
print(df_pool['categoria'].value_counts())

Mounted at /content/drive
Unificado oficial: 1400
categoria
Backend           200
Bases de Datos    200
Cloud             200
Data Science      200
DevOps            200
Frontend          200
Mobile            200
Name: count, dtype: int64

Pool: 1693
categoria
Backend           250
Frontend          250
Data Science      250
Cloud             248
Bases de Datos    240
DevOps            238
Mobile            217
Name: count, dtype: int64
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Unificado oficial: 1400
categoria
Backend           200
Bases de Datos    200
Cloud             200
Data Science      200
DevOps            200
Frontend          200
Mobile            200
Name: count, dtype: int64

Pool de fuentes: 1694
categoria
Backend           250
Frontend          250
Data Science      250
Cloud             248
Bases de Datos    240
DevOps            238
Mobile            218
Name: count, dtype: int64


## Ítem 1: Más ejemplos en las categorías con F1 bajo

### Objetivo
Dar más evidencia de entrenamiento a Backend, Cloud, Data Science y Frontend
(las categorías con F1 0.53/0.66/0.70/0.72 en el baseline).

### ¿Cómo funciona?
- Se deduplica el pool entre fuentes por título+texto normalizados.
- Se eleva el tope de balanceo de 200 a 250 solo en esas cuatro categorías.

### Resultado esperado
~1.600 filas: 250/250/250/248 en las flojas y 200 en el resto.

In [ ]:
def norm(s): return s.str.lower().str.strip().str.replace(r'\s+', ' ', regex=True)   # se normaliza para la clave de dedup

df_pool['_t'] = norm(df_pool['titulo']) + ' | ' + norm(df_pool['texto'])   # se crea la clave temporal título|texto
df_pool = df_pool.drop_duplicates(subset='_t')   # se eliminan duplicados entre fuentes

F1_BAJO = ['Backend', 'Cloud', 'Data Science', 'Frontend']   # categorías con F1 bajo detectado en la Semana 2
TOPE = {cat: (250 if cat in F1_BAJO else 200)                # se define el tope por categoría:
        for cat in df_pool['categoria'].unique()}            # 250 en las flojas, 200 en el resto

df_ampliado = pd.concat([                                    # se unen los muestreos por categoría
    g.sample(min(len(g), TOPE[cat]), random_state=42)        # muestreo sin reemplazo y determinista (reproducible)
    for cat, g in df_pool.groupby('categoria')               # se recorre cada categoría con su grupo
], ignore_index=True).drop(columns=['_t'])                   # se descarta la clave temporal
print("Dataset ampliado (Ítem 1):")                          # se reporta el resultado
print(df_ampliado['categoria'].value_counts())               # se muestra la distribución final

Dataset ampliado (Ítem 1):
categoria
Backend           250
Frontend          250
Data Science      250
Cloud             248
Bases de Datos    200
DevOps            200
Mobile            200
Name: count, dtype: int64


## Ítem 2: Reentrenar idéntico al baseline y comparar

### Objetivo
Aislar el efecto del dato: mismo pipeline, misma semilla, mismas métricas.

### ¿Cómo funciona?
- Split 80/20 estratificado con semilla 42 (igual que el baseline).
- TF-IDF (3.000 features, mismo token_pattern) + Regresión Logística (max_iter=1000).
- Se compara F1 por categoría, accuracy y F1 macro contra el baseline del PR #1.

In [ ]:
# Métricas publicadas del baseline (PR #1) contra las que se compara
BASELINE_F1 = {'Backend':0.53,'Bases de Datos':0.80,'Cloud':0.66,'Data Science':0.70,
               'DevOps':0.73,'Frontend':0.72,'Mobile':0.81}

def entrenar_y_evaluar(df, nombre, semilla=42):
    """Se entrena el pipeline idéntico al baseline y se devuelven métricas y artefactos."""
    df = df.copy()                                # se trabaja sobre una copia
    df['texto_limpio'] = df['texto'].apply(limpiar_texto)   # se aplica la limpieza canónica
    X_tr, X_te, y_tr, y_te = train_test_split(    # se hace el split estratificado
        df['texto_limpio'], df['categoria'],      # mismas columnas que el baseline
        test_size=0.2, random_state=semilla, stratify=df['categoria'])   # misma proporción y semilla
    vec = TfidfVectorizer(max_features=3000,      # mismos hiperparámetros del baseline
                          token_pattern=r"(?u)\b(?=\w*[^\W\d_])\w{2,}\b")   # conserva términos como s3, ipv6
    Xtr = vec.fit_transform(X_tr)                 # se ajusta el vectorizador solo con train
    Xte = vec.transform(X_te)                     # se transforma test sin filtrar información
    clf = LogisticRegression(max_iter=1000)       # mismo clasificador del baseline
    clf.fit(Xtr, y_tr)                            # se entrena con las etiquetas de train
    pred = clf.predict(Xte)                       # se predice sobre test
    rep = classification_report(y_te, pred, output_dict=True)   # se obtiene el reporte completo
    f1s = {c: round(rep[c]['f1-score'],2) for c in BASELINE_F1}   # se extrae el F1 por categoría
    acc = accuracy_score(y_te, pred)              # se calcula el accuracy
    f1m = f1_score(y_te, pred, average='macro')   # se calcula el F1 macro
    print(f"--- {nombre}: accuracy {acc:.2f} | F1 macro {f1m:.2f}")   # se imprime el resumen
    return clf, vec, f1s, acc, f1m                # se devuelven artefactos y métricas

def comparar(etq, f1s, acc, f1m):
    """Se imprime la tabla antes/después por categoría contra el baseline."""
    print(pd.DataFrame([{'categoría':c,'baseline':BASELINE_F1[c],'nuevo':f1s[c],   # se arma la tabla
          'Δ':round(f1s[c]-BASELINE_F1[c],2)} for c in BASELINE_F1]).to_string(index=False))
    print(f"[{etq}] Accuracy 0.71 → {acc:.2f} | F1 macro 0.71 → {f1m:.2f}\n")   # se imprime el delta global

## Ejecución: sanity + mejora + exportación SIN pisar el baseline:

In [ ]:
# A) SANITY CHECK: debe reproducir ≈0.71 (±0.03); valida que el arnés es fiel
_,_,f1_a,acc_a,f1m_a = entrenar_y_evaluar(df_actual, "SANITY (unificado oficial)")
comparar("sanity", f1_a, acc_a, f1m_a)            # se imprime la tabla de fidelidad

# B) MODELO MEJORADO: Coursera v2 + OpenAlex v2 + ejemplos extra en las flojas
clf_m, vec_m, f1_b, acc_b, f1m_b = entrenar_y_evaluar(df_ampliado, "MEJORA (v2 + ejemplos extra)")
comparar("mejora", f1_b, acc_b, f1m_b)            # se imprime la tabla del ítem 2

# C) Se exportan los artefactos con nombres NUEVOS (el equipo decide cuándo hacer el swap)
CARP = DRIVE / 'modelo_mejorado'; CARP.mkdir(exist_ok=True)   # se crea la carpeta si no existe
joblib.dump(clf_m, CARP/'modelo_mejorado.pkl')    # se guarda el modelo sin pisar el baseline
joblib.dump(vec_m, CARP/'vectorizer_mejorado.pkl')   # se guarda el vectorizador sin pisar el baseline
print("✅ Artefactos guardados en modelo_mejorado/ (baseline intacto)")   # se confirma

--- SANITY (unificado oficial): accuracy 0.79 | F1 macro 0.79
     categoría  baseline  nuevo     Δ
       Backend      0.53   0.65  0.12
Bases de Datos      0.80   0.75 -0.05
         Cloud      0.66   0.76  0.10
  Data Science      0.70   0.81  0.11
        DevOps      0.73   0.84  0.11
      Frontend      0.72   0.82  0.10
        Mobile      0.81   0.91  0.10
[sanity] Accuracy 0.71 → 0.79 | F1 macro 0.71 → 0.79

--- MEJORA (v2 + ejemplos extra): accuracy 0.80 | F1 macro 0.81
     categoría  baseline  nuevo    Δ
       Backend      0.53   0.67 0.14
Bases de Datos      0.80   0.85 0.05
         Cloud      0.66   0.83 0.17
  Data Science      0.70   0.84 0.14
        DevOps      0.73   0.76 0.03
      Frontend      0.72   0.80 0.08
        Mobile      0.81   0.91 0.10
[mejora] Accuracy 0.71 → 0.80 | F1 macro 0.71 → 0.81

✅ Artefactos guardados en modelo_mejorado/ (baseline intacto)


In [ ]:
import pandas as pd

base = '/content/drive/MyDrive/Datasets_TechMind/1 Semana - con el Dataset final con todo incluido - Repo GitHub/procesados/'
df_uni = pd.read_csv(base + 'dataset_FINAL_UNIFICADO_techmind.csv')

# Cargamos tu versión corregida original desde tu carpeta personal
ruta_laura = '/content/drive/MyDrive/Datasets_TechMind/Code_Laura/Datos_procesados/dataset_FINAL_stackexchange_corregido.csv'
df_laura = pd.read_csv(ruta_laura)

# Creamos sets con los primeros 100 caracteres de cada texto para comparar de forma rápida
uni_txt = set(df_uni['texto'].str[:100].dropna())
laura_txt = set(df_laura['texto'].str[:100].dropna())

# ¿Cuántos textos del unificado provienen EXACTAMENTE de tu trabajo en StackExchange?
coincidencias = len(uni_txt & laura_txt)

print(f"Textos en el Unificado Oficial: {len(uni_txt)}")
print(f"Textos en tu StackExchange Corregido (Laura): {len(laura_txt)}")
print(f"✅ Coincidencias exactas (Unificado usa tus respuestas): {coincidencias}")

# Como el unificado tiene 1400 filas y tu SE tiene 700, esperamos que al menos las 700
# (o un poco menos por deduplicación con otras fuentes) estén ahí.
if coincidencias > 500:
    print("\n🎉 CONFIRMADO: El unificado oficial se construyó usando TUS respuestas aceptadas, no las preguntas viejas.")

Textos en el Unificado Oficial: 1385
Textos en tu StackExchange Corregido (Laura): 700
✅ Coincidencias exactas (Unificado usa tus respuestas): 567

🎉 CONFIRMADO: El unificado oficial se construyó usando TUS respuestas aceptadas, no las preguntas viejas.
